# WSMTE — 01_data_prep.ipynb
**LOCAL PC** | Steps 2–8 from DATA_PIPELINE.md  
Loads finbert outputs + price data, aggregates to daily, merges by trading date.

**Prerequisites**: Run `notebooks/03_finbert_inference.ipynb` on Kaggle GPU first, then
download the 3 output CSVs into `data/finbert_outputs/`.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # WSMTE root on Kaggle; no-op locally

import pandas as pd
from config.config import CONFIG

## Step 2 — Load Raw Datasets

In [3]:
# ── Price data ──────────────────────────────────────────────────────────────
price_df = pd.read_csv(CONFIG['raw_data_dir'] + 'nifty50_ohlcv.csv')
price_df['date'] = pd.to_datetime(price_df['Date']).dt.date
price_df = price_df.sort_values('date').reset_index(drop=True)
print(f"Price data: {price_df.shape} | {price_df['date'].min()} to {price_df['date'].max()}")

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/nifty50_ohlcv.csv'

In [ ]:
# ── Kotekar dataset (company-level) ─────────────────────────────────────────
# Columns: datePublished, company, symbol, headline, description,
#          articleBody, tags, author, url
kotekar = pd.read_csv(CONFIG['raw_data_dir'] + 'kotekar_news.csv')
kotekar['date'] = pd.to_datetime(kotekar[CONFIG['kotekar_date_col']]).dt.date
kotekar = kotekar[
    (kotekar['date'] >= pd.to_datetime(CONFIG['start_date']).date()) &
    (kotekar['date'] <= pd.to_datetime(CONFIG['end_date']).date())
]
print(f"Kotekar: {kotekar.shape}")

In [ ]:
# ── Kaggle Dataset 1 (Jan 2020 – Apr 15, 2021, market-level) ────────────────────
# Columns: Date (DD/MM/YY), Title, URL, sentiment, confidence
# NOTE: existing sentiment/confidence ignored — FinBERT reruns for consistency
kaggle1 = pd.read_csv(CONFIG['raw_data_dir'] + 'kaggle_news_1.csv')
kaggle1['date'] = pd.to_datetime(kaggle1[CONFIG['kaggle1_date_col']], format='%d/%m/%y').dt.date
kaggle1 = kaggle1[
    (kaggle1['date'] >= pd.to_datetime(CONFIG['start_date']).date()) &
    (kaggle1['date'] <= pd.to_datetime('2021-04-15').date())
]
print(f"Kaggle1: {kaggle1.shape} | {kaggle1['date'].min()} to {kaggle1['date'].max()}")

In [ ]:
# ── Kaggle Dataset 2 (Jan 2022 – Apr 2024, market-level) ────────────────────
# Columns: Archive, Date (DD-MM-YYYY), Headline, Headline link
kaggle2 = pd.read_csv(CONFIG['raw_data_dir'] + 'kaggle_news_2.csv')
kaggle2['date'] = pd.to_datetime(kaggle2[CONFIG['kaggle2_date_col']], format='%d-%m-%Y').dt.date
kaggle2 = kaggle2[
    (kaggle2['date'] >= pd.to_datetime('2022-01-01').date()) &
    (kaggle2['date'] <= pd.to_datetime(CONFIG['end_date']).date())
]
print(f"Kaggle2: {kaggle2.shape} | {kaggle2['date'].min()} to {kaggle2['date'].max()}")

## Steps 3–6 — Load FinBERT / mDeBERTa Outputs
(Generated by `notebooks/03_finbert_inference.ipynb` on Kaggle GPU)

In [ ]:
# ── kotekar_sentiment.csv ───────────────────────────────────────────────────
# Cols: date, company, symbol, polarity_company, subjectivity
kotekar_sent = pd.read_csv(CONFIG['kotekar_sentiment_file'])
kotekar_sent['date'] = pd.to_datetime(kotekar_sent['date']).dt.date
print(f"Kotekar sentiment: {kotekar_sent.shape}")
print(f"  polarity_company range: {kotekar_sent['polarity_company'].min():.3f} to {kotekar_sent['polarity_company'].max():.3f}")
print(f"  subjectivity range:     {kotekar_sent['subjectivity'].min():.3f} to {kotekar_sent['subjectivity'].max():.3f}")

In [ ]:
# ── kaggle1_polarity.csv ────────────────────────────────────────────────────
# Cols: date, polarity_market
kaggle1_pol = pd.read_csv(CONFIG['kaggle1_polarity_file'])
kaggle1_pol['date'] = pd.to_datetime(kaggle1_pol['date']).dt.date
print(f"Kaggle1 polarity: {kaggle1_pol.shape}")

# ── kaggle2_polarity.csv ────────────────────────────────────────────────────
kaggle2_pol = pd.read_csv(CONFIG['kaggle2_polarity_file'])
kaggle2_pol['date'] = pd.to_datetime(kaggle2_pol['date']).dt.date
print(f"Kaggle2 polarity: {kaggle2_pol.shape}")

## Step 7 — Aggregate All Sentiment to Daily Level

In [ ]:
# ── Company-level: mean per trading day ─────────────────────────────────────
company_daily = kotekar_sent.groupby('date').agg(
    polarity_company=('polarity_company', 'mean'),
    subjectivity=('subjectivity', 'mean')
).reset_index()
print(f"Company daily: {company_daily.shape}")

# ── Market-level: combine Kaggle 1 + 2, no date overlap ─────────────────────
# Kaggle1 covers up to Apr 15, 2021, Kaggle2 from Jan 2022; gap May–Dec 2021
market_combined = pd.concat([
    kaggle1_pol[['date', 'polarity_market']],
    kaggle2_pol[['date', 'polarity_market']]
]).sort_values('date').reset_index(drop=True)

market_daily = market_combined.groupby('date').agg(
    polarity_market=('polarity_market', 'mean')
).reset_index()
print(f"Market daily: {market_daily.shape}")

## Step 8 — Merge All Sources by Trading Date

In [ ]:
# Left join on price data (trading days are the anchor)
df = price_df.merge(company_daily, on='date', how='left')
df = df.merge(market_daily, on='date', how='left')

# Fill gap period May–Dec 2021 + any other missing
# polarity_market = 0 (neutral), polarity_company = 0, subjectivity = 0.5
df['polarity_market']  = df['polarity_market'].fillna(CONFIG['missing_polarity'])
df['polarity_company'] = df['polarity_company'].fillna(CONFIG['missing_polarity'])
df['subjectivity']     = df['subjectivity'].fillna(CONFIG['missing_subjectivity'])

df = df.sort_values('date').reset_index(drop=True)
print(f"Merged shape: {df.shape}")
print(f"Date range:   {df['date'].min()} to {df['date'].max()}")
print(f"Missing values:
{df[['polarity_market','polarity_company','subjectivity']].isnull().sum()}")

In [ ]:
# Verify gap period May–Dec 2021 has polarity_market = 0
gap_start = pd.to_datetime(CONFIG['gap_start']).date()
gap_end   = pd.to_datetime(CONFIG['gap_end']).date()
gap = df[(df['date'] >= gap_start) & (df['date'] <= gap_end)]
assert (gap['polarity_market'] == 0.0).all(), "Gap period fill failed!"
print(f"Gap period ({gap_start} to {gap_end}): {len(gap)} trading days, polarity_market = 0 ✓")

In [ ]:
df.to_csv(CONFIG['processed_data_dir'] + 'merged_data.csv', index=False)
print("Saved → data/processed/merged_data.csv")